In [1]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

from preprocessing import build_preprocessor, load_and_split

In [2]:
x_train, x_test, y_train, y_test = load_and_split("../data/student-mat.csv")


In [3]:
x_train.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,study_per_absence,failure_impact
181,GP,M,16,U,GT3,T,3,3,services,other,...,2,3,1,2,3,2,12,13,0.666667,0
194,GP,M,16,U,GT3,T,2,3,other,other,...,3,3,1,1,3,0,13,14,1.000000,0
173,GP,F,16,U,GT3,T,1,3,at_home,services,...,3,5,1,1,3,0,8,7,2.000000,0
63,GP,F,16,U,GT3,T,4,3,teacher,health,...,4,4,2,4,4,2,10,9,1.000000,0
253,GP,M,16,R,GT3,T,2,1,other,other,...,3,2,1,3,3,0,8,9,1.000000,0


In [4]:
preprocessor = build_preprocessor(x_train)

In [5]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

In [6]:
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    # CV
    cv_scores = cross_val_score(pipeline, x_train, y_train, cv=5, scoring='r2')
    cv_r2 = cv_scores.mean()

    # Train + Test
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    test_r2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5

    results.append({
        "Model": name,
        "CV R2": round(cv_r2, 4),
        "Test R2": round(test_r2, 4),
        "RMSE": round(rmse, 4)
    })

df_results = pd.DataFrame(results).sort_values(by="Test R2", ascending=False)
display(df_results)

,Model,CV R2,Test R2,RMSE
1,Random Forest,0.9011,0.8228,1.9061
2,Gradient Boosting,0.8950,0.8159,1.9430
0,Linear Regression,0.8439,0.7620,2.2093


Cross-validation giúp đánh giá độ ổn định, trong khi test set phản ánh hiệu suất thực tế của mô hình trên dữ liệu chưa thấy.
Random Forest tốt nhất cho bài toán này

In [ ]:
# Danh sách các giá trị cần thử
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10]
}

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(x_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(x_test)

print("Final R2:", r2_score(y_test, y_pred))

Final R2: 0.8228146031944488


In [ ]:
import joblib
import os

model_export_path = "../models/best_model.pkl"

os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, model_export_path)

print(f"Đã lưu model thành công tại: {model_export_path}")

Đã lưu model tại models/best_model.pkl
